# Chapter 16: Describing Data With Statistics

The programs from Chapter 16 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## Typical Values and Spread

`describe_stats.py`

In [1]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv")
value = orders["quantity"] * orders["unit_price"] * (1 - orders["discount"])

q1, q3 = value.quantile([0.25, 0.75])
print(f"mean     {value.mean():8.2f}")
print(f"median   {value.median():8.2f}")
print(f"std      {value.std():8.2f}")
print(f"IQR      {q3 - q1:8.2f}  (middle half: {q1:.0f} to {q3:.0f})")
print(f"skew     {value.skew():8.2f}  (positive: a long right tail)")
print(f"top 1%   {value.quantile(0.99):8.2f} and above")

mean        65.10
median      49.50
std         57.67
IQR         51.20  (middle half: 29 to 80)
skew         2.94  (positive: a long right tail)
top 1%     288.00 and above


## Correlation, and Looking for Delays

`correlation.py`

In [2]:
import pandas as pd

air = pd.read_csv("../data/air_quality.csv", parse_dates=["date"])
visits = pd.read_csv("../data/clinic_visits.csv", parse_dates=["date"])
city = (
    air.merge(visits, on=["date", "city"])
    .query("city == 'Brightwater'")
    .set_index("date")
)

for lag in range(5):
    r = city["visits"].corr(city["pm25"].shift(lag))
    print(f"pollution {lag} days earlier: r = {r:.2f}")

pollution 0 days earlier: r = 0.43
pollution 1 days earlier: r = 0.50
pollution 2 days earlier: r = 0.76
pollution 3 days earlier: r = 0.48
pollution 4 days earlier: r = 0.43


## Simpson's Paradox

`simpson.py`

In [3]:
import pandas as pd

# Two support teams; tickets are easy or hard.
tickets = pd.DataFrame(
    {
        "team": ["Red", "Red", "Blue", "Blue"],
        "kind": ["easy", "hard", "easy", "hard"],
        "tickets": [90, 10, 20, 80],
        "solved": [81, 5, 19, 48],
    }
)

overall = tickets.groupby("team")[["solved", "tickets"]].sum()
print((overall["solved"] / overall["tickets"]).round(2))
print()
tickets["rate"] = (tickets["solved"] / tickets["tickets"]).round(2)
print(tickets.pivot(index="kind", columns="team", values="rate"))

team
Blue    0.67
Red     0.86
dtype: float64

team  Blue  Red
kind           
easy  0.95  0.9
hard  0.60  0.5
